# Aegis evaluation on Colab

Run on Colab Pro (T4 or A100). Cells run top-to-bottom and produce result JSONs
at `eval/results/<predictor>/` on the Colab VM. After running, copy that folder
to Google Drive (cell 16), download to your local repo, then re-inject the README.

What this notebook runs:
- **Aegis** — your fine-tuned LoRA adapter (GPU-bound, ~15-60 min)
- **GPT-4o-mini** — optional, needs `OPENAI_API_KEY`
- **Gemini Flash** — optional, needs `GOOGLE_API_KEY`
- **Cross-LLM-judge calibration** on Aegis — needs BOTH API keys

Each baseline goes through the same `Predictor` interface as Presidio so
metrics are directly comparable.

In [1]:
!python --version
!nvidia-smi -L

Python 3.12.13
GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-a80a7f74-5d59-8513-e3a2-fa2949cfa5f7)


## 1. Setup

In [3]:
import os

REPO_URL = "https://github.com/Venkata1345/Aegis-finetuning.git"
REPO_DIR = "/content/fine-tuning"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

Cloning into '/content/fine-tuning'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 52 (delta 1), reused 52 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 102.16 KiB | 17.03 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/fine-tuning


In [4]:
!pip install -q -r requirements.txt
!pip install -q -e .

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.2/254.2 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 131.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 118.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 15.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyopenssl 24.2.1 requires cryptography<44,>=41.0.5, but you have cryptography 48.0.0 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 48.0.0 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ...

In [5]:
# Fill in your real values; do NOT commit them anywhere.
import os
os.environ["HF_TOKEN"]  = "hf_..."                              # required for adapter pull
os.environ["AEGIS_ADAPTER_REPO"] = "Abhishek4545/aegis-qwen-7b-lora"     # your trained adapter
os.environ["OPENAI_API_KEY"]     = "sk-proj-..."                         # optional (gates GPT-4o-mini + judge)
os.environ["GOOGLE_API_KEY"]     = "AIza..."                              # optional (gates Gemini Flash + judge)

In [6]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [7]:
# Only needed on a fresh VM where the processed data isn't present.
import pathlib
if not pathlib.Path("data/processed/validation.jsonl").exists():
    !python -m data.download
    !python -m data.convert
    !python -m data.format
else:
    print("Processed data already present, skipping rebuild.")

Loading ai4privacy/pii-masking-300k from Hugging Face Hub...
README.md: 16.7kB [00:00, 19.3MB/s]
data/train/1english_openpii_30k.jsonl: 100% 103M/103M [00:02<00:00, 46.1MB/s]
data/train/dutch_openpii_28k.jsonl: 100% 102M/102M [00:01<00:00, 84.6MB/s]
data/train/french_openpii_31k.jsonl: 100% 114M/114M [00:01<00:00, 62.9MB/s]
data/train/german_openpii_30k.jsonl: 100% 108M/108M [00:01<00:00, 76.2MB/s]
data/train/italian_openpii_29k.jsonl: 100% 104M/104M [00:01<00:00, 85.6MB/s]
data/train/spanish_openpii_29k.jsonl: 100% 102M/102M [00:01<00:00, 84.0MB/s]
data/validation/1english_openpii_8k.json(…): 100% 27.3M/27.3M [00:00<00:00, 44.7MB/s]
data/validation/dutch_openpii_7k.jsonl: 100% 27.0M/27.0M [00:00<00:00, 44.1MB/s]
data/validation/french_openpii_8k.jsonl: 100% 30.7M/30.7M [00:00<00:00, 37.8MB/s]
data/validation/german_openpii_8k.jsonl: 100% 29.2M/29.2M [00:00<00:00, 71.1MB/s]
data/validation/italian_openpiii_8k.json(…): 100% 28.3M/28.3M [00:00<00:00, 46.2MB/s]
data/validation/spanish_ope

## 2. Aegis eval — the main event

GPU-bound. ~15-30 min on A100, ~30-60 min on T4 for all three datasets
(7,946 main + 50 heldout + 50 adversarial). Output lands in
`eval/results/aegis/{main,heldout,adversarial}.json` plus per-instance logs.

In [8]:
!python -m eval.run_baseline aegis

usage: run_baseline.py [-h] [--smoke] [--max-main MAX_MAIN]
                       {presidio,openai,gemini,qwen-base}
run_baseline.py: error: argument predictor: invalid choice: 'aegis' (choose from presidio, openai, gemini, qwen-base)


## 3. Commercial-API baselines (optional)

Cost: ~$3-5 across both APIs for ~8,000 calls each. Skip cells whose API key
you don't have — the others will still run.

In [ ]:
# GPT-4o-mini — needs OPENAI_API_KEY in cell 5
!python -m eval.run_baseline openai

In [ ]:
# Gemini Flash — needs GOOGLE_API_KEY in cell 5
!python -m eval.run_baseline gemini

## 4. Cross-LLM-judge calibration on Aegis

Two judges (GPT-4o + Gemini 2.5 Pro) score the same 50 adversarial outputs
on a 1-5 rubric. Pearson r between them is the reliability proxy. Target
r ≥ 0.6; below that the rubric is iterated.

**Requires BOTH `OPENAI_API_KEY` and `GOOGLE_API_KEY`.** Cost: ~$0.10-0.20.

In [ ]:
!python -m eval.calibration --predictor aegis --sample 50

## 5. Copy results to Drive

Then download the `aegis-eval-results/` folder from Drive to your local repo
under `eval/results/`. The local harness will pick them up.

In [ ]:
import os, shutil

src = "/content/fine-tuning/eval/results"
dst = "/content/drive/MyDrive/aegis-eval-results"
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)
print(f"Copied {src} -> {dst}")
print("Now download the folder from Drive and unpack into your local eval/results/.")

## Done

On your local machine, after downloading `aegis-eval-results/` from Drive into
`eval/results/`:

```bash
python -m eval.harness --inject README.md
```

That regenerates the comparison table with Aegis (and any commercial baselines
you ran) included. The delta from Presidio (already in README) → Aegis is the
project's headline result.